# ECA-DNet: Efficient Channel Attention Dehazing Network
## Full Training Pipeline — 
**Datasets Required (Add via + Add Input):**
1. `kmljts/reside-6k`
2. `balraj98/indoor-training-set-its-residestandard`
3. `balraj98/synthetic-objective-testing-set-sots-reside`

**Settings:** GPU P100 | Persistence: Files | Internet: ON

In [ ]:

import os, cv2, json, time, random, warnings, gc, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers, callbacks
from tensorflow.keras.mixed_precision import set_global_policy

warnings.filterwarnings('ignore')
# set_global_policy('mixed_float16')
tf.config.optimizer.set_jit(False) # Enable XLA compilation

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {len(gpus)}")
if gpus:
    print(f"GPU name   : {gpus[0].name}")
# print(f"Precision  : {tf.keras.mixed_precision.global_policy().name}")

In [ ]:

import os

# Data Configuration
IMG_SIZE         = 256
CHANNELS         = 3

# Architecture
ENC_FILTERS      = [32, 64, 128, 256, 512]
DEC_FILTERS      = [256, 128, 64, 32]
ECA_KERNEL       = 3

# Regularization (NO L2 since AdamW handles weight decay)
SPATIAL_DROP     = 0.05
BOTTLENECK_DROP  = 0.10

# Training Parameters
BATCH_SIZE       = 16
BATCH_SIZE_S3    = 8

# Multi-Stage Epoch Schedule (110 Epochs Total)
LR_S1, EPOCHS_S1, WARMUP_S1 = 1e-3, 60, 5
LR_S2, EPOCHS_S2, WARMUP_S2 = 1e-4, 30, 2
LR_S3, EPOCHS_S3            = 5e-5, 20

PATIENCE_ES      = 15
PATIENCE_LR      = 7
MIN_LR           = 1e-7

#  Loss weights (VGG Perceptual added)++
W_MAE, W_SSIM, W_EDGE, W_FREQ, W_PERCEP = 0.35, 0.20, 0.10, 0.10, 0.25

# Augmentation
AUG_BRIGHTNESS   = 0.08
AUG_NOISE_STD    = 0.01
AUG_CONTRAST     = 0.10

# Paths
SAVE_DIR    = '/kaggle/working'
MODEL_DIR   = f'{SAVE_DIR}/models'
HISTORY_DIR = f'{SAVE_DIR}/history'
PLOT_DIR    = f'{SAVE_DIR}/plots'
METRIC_DIR  = f'{SAVE_DIR}/metrics'
RESULT_DIR  = f'{SAVE_DIR}/results'
LOG_DIR     = f'{SAVE_DIR}/logs'

for d in [MODEL_DIR, HISTORY_DIR, PLOT_DIR, METRIC_DIR, RESULT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Configuration loaded")
print(f"  Enc filters : {ENC_FILTERS}")
print(f"  Total epochs: {EPOCHS_S1+EPOCHS_S2+EPOCHS_S3}")


In [ ]:


def find_dir(base, candidates):
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p) and len(os.listdir(p)) > 0:
            return p
    # Try one level deeper
    if os.path.isdir(base):
        for sub in os.listdir(base):
            sp = os.path.join(base, sub)
            if os.path.isdir(sp):
                for c in candidates:
                    p = os.path.join(sp, c)
                    if os.path.isdir(p) and len(os.listdir(p)) > 0:
                        return p
    return None

R6K = '/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K'
R6K_TRAIN_HAZY = find_dir(R6K, ['train/hazy','Train/hazy','train/Hazy'])
R6K_TRAIN_GT   = find_dir(R6K, ['train/GT','train/gt','train/clear','Train/GT'])
R6K_TEST_HAZY  = find_dir(R6K, ['test/hazy','Test/hazy','test/Hazy'])
R6K_TEST_GT    = find_dir(R6K, ['test/GT','test/gt','test/clear','Test/GT'])

ITS  = '/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard'
ITS_HAZY = find_dir(ITS, ['hazy','Hazy','ITS/hazy','data/hazy','train/hazy'])
ITS_GT   = find_dir(ITS, ['clear','Clear','GT','gt','clean','ITS/clear','data/clear','train/clear','train/GT'])

SOTS = '/kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside'
SOTS_IN_HAZY  = find_dir(SOTS, ['indoor/hazy','SOTS/indoor/hazy','sots/indoor/hazy'])
SOTS_IN_GT    = find_dir(SOTS, ['indoor/clear','indoor/gt','indoor/clear','SOTS/indoor/GT','SOTS/indoor/gt'])
SOTS_OUT_HAZY = find_dir(SOTS, ['outdoor/hazy','SOTS/outdoor/hazy','sots/outdoor/hazy'])
SOTS_OUT_GT   = find_dir(SOTS, ['outdoor/clear','outdoor/Clear','outdoor/clear','SOTS/outdoor/GT'])

print("="*65)
print("DATASET PATH VERIFICATION")
print("="*65)
for name, path in [
    ('R6K Train Hazy', R6K_TRAIN_HAZY), ('R6K Train GT', R6K_TRAIN_GT),
    ('R6K Test Hazy',  R6K_TEST_HAZY),  ('R6K Test GT',  R6K_TEST_GT),
    ('ITS Hazy',       ITS_HAZY),        ('ITS GT',       ITS_GT),
    ('SOTS In Hazy',   SOTS_IN_HAZY),   ('SOTS In GT',   SOTS_IN_GT),
    ('SOTS Out Hazy',  SOTS_OUT_HAZY),  ('SOTS Out GT',  SOTS_OUT_GT),
]:
    if path:
        n = len(os.listdir(path))
        print(f"  OK  {name:<18} {path}  [{n} files]")
    else:
        print(f"  XX  {name:<18} NOT FOUND")
print("="*65)

In [ ]:


def preprocess_hazy(img):
    img = np.power(np.clip(img, 0, 1), 0.9).astype(np.float32)
    for c in range(3):
        lo, hi = img[:,:,c].min(), img[:,:,c].max()
        if hi > lo + 1e-6:
            img[:,:,c] = (img[:,:,c] - lo) / (hi - lo)
    return np.clip(img, 0, 1).astype(np.float32)

def load_image_pair(hazy_path, gt_path, size=IMG_SIZE):
    hazy = cv2.imread(str(hazy_path))
    gt   = cv2.imread(str(gt_path))
    if hazy is None or gt is None:
        return None, None
    hazy = cv2.cvtColor(hazy, cv2.COLOR_BGR2RGB)
    gt   = cv2.cvtColor(gt,   cv2.COLOR_BGR2RGB)
    hazy = cv2.resize(hazy, (size, size)).astype(np.float32) / 255.0
    gt   = cv2.resize(gt,   (size, size)).astype(np.float32) / 255.0
    hazy = preprocess_hazy(hazy)
    return hazy, gt

def get_gt_path(hazy_name, gt_dir):
    stem = Path(hazy_name).stem
    candidates = [stem, stem.split('_')[0], stem.replace('_hazy',''),
                  stem.replace('_fog',''), stem.replace('_synthetic','')]
    for cand in candidates:
        for ext in ['.png','.jpg','.jpeg','.PNG','.JPG','.bmp']:
            p = os.path.join(gt_dir, cand + ext)
            if os.path.exists(p):
                return p
    return None

def collect_pairs(hazy_dir, gt_dir, tag='', limit=None):
    if not hazy_dir or not gt_dir:
        return []
    files = sorted(os.listdir(hazy_dir))
    if limit:
        files = files[:limit]
    pairs = []
    for hf in files:
        if not hf.lower().endswith(('.png','.jpg','.jpeg','.bmp')):
            continue
        gp = get_gt_path(hf, gt_dir)
        if gp:
            pairs.append((os.path.join(hazy_dir, hf), gp))
    skip = len(files) - len(pairs)
    print(f"  {tag:<22}: {len(pairs):>5} pairs  ({skip} skipped)")
    return pairs

print("Collecting training pairs...")
r6k_pairs = collect_pairs(R6K_TRAIN_HAZY, R6K_TRAIN_GT, 'RESIDE-6K train')
its_pairs  = collect_pairs(ITS_HAZY, ITS_GT, 'ITS indoor')

all_train = r6k_pairs + its_pairs
random.shuffle(all_train)
print(f"\nTotal training pairs : {len(all_train)}")

hazy_paths = [p[0] for p in all_train]
gt_paths   = [p[1] for p in all_train]

tr_h, va_h, tr_g, va_g = train_test_split(
    hazy_paths, gt_paths, test_size=0.10, random_state=SEED, shuffle=True)
print(f"Train : {len(tr_h)}  |  Val : {len(va_h)}")

In [ ]:



def load_pair_tf(hazy_path, gt_path):
    def _py_load(hp, gp):
        hp = hp.numpy().decode('utf-8')
        gp = gp.numpy().decode('utf-8')
        h, g = load_image_pair(hp, gp)
        if h is None:
            h = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float32)
            g = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float32)
        return h, g
    hazy, gt = tf.py_function(_py_load, [hazy_path, gt_path], [tf.float32, tf.float32])
    hazy.set_shape([IMG_SIZE, IMG_SIZE, 3])
    gt.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return hazy, gt

def augment_pair(hazy, gt):
    # Spatial (both)
    do_h = tf.random.uniform(()) > 0.5
    hazy = tf.cond(do_h, lambda: tf.image.flip_left_right(hazy), lambda: hazy)
    gt   = tf.cond(do_h, lambda: tf.image.flip_left_right(gt),   lambda: gt)
    do_v = tf.random.uniform(()) > 0.5
    hazy = tf.cond(do_v, lambda: tf.image.flip_up_down(hazy), lambda: hazy)
    gt   = tf.cond(do_v, lambda: tf.image.flip_up_down(gt),   lambda: gt)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    hazy = tf.image.rot90(hazy, k)
    gt   = tf.image.rot90(gt, k)
    # Photometric (hazy only)
    hazy = tf.image.random_brightness(hazy, max_delta=AUG_BRIGHTNESS)
    hazy = tf.image.random_contrast(hazy, 1-AUG_CONTRAST, 1+AUG_CONTRAST)
    noise = tf.random.normal(tf.shape(hazy), stddev=AUG_NOISE_STD)
    hazy = hazy + noise
    hazy = tf.clip_by_value(hazy, 0.0, 1.0)
    gt   = tf.clip_by_value(gt, 0.0, 1.0)
    return hazy, gt

def build_dataset(hazy_list, gt_list, augment=True, batch=BATCH_SIZE, shuffle=True, cache_ds=False):
    ds = tf.data.Dataset.from_tensor_slices((hazy_list, gt_list))
    if shuffle:
        ds = ds.shuffle(min(len(hazy_list), 3000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_pair_tf, num_parallel_calls=tf.data.AUTOTUNE)
    if cache_ds:
        ds = ds.cache()
    if augment:
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print("Building tf.data pipelines...")
train_ds = build_dataset(tr_h, tr_g, augment=True, batch=BATCH_SIZE)
val_ds   = build_dataset(va_h, va_g, augment=False, batch=BATCH_SIZE, shuffle=False, cache_ds=True)

STEPS_PER_EPOCH = len(tr_h) // BATCH_SIZE
VAL_STEPS = len(va_h) // BATCH_SIZE
print(f"Train steps/epoch : {STEPS_PER_EPOCH}")
print(f"Val   steps/epoch : {VAL_STEPS}")

for hb, gb in train_ds.take(1):
    print(f"Hazy batch : {hb.shape}  [{hb.numpy().min():.3f}, {hb.numpy().max():.3f}]")
    print(f"GT   batch : {gb.shape}  [{gb.numpy().min():.3f}, {gb.numpy().max():.3f}]")

## Model Architecture — Building Blocks

In [ ]:

class ECABlock(layers.Layer):
    def __init__(self, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.conv = layers.Conv1D(1, self.kernel_size, padding='same', use_bias=False)
        self.gap = layers.GlobalAveragePooling2D()
    def call(self, x):
        # x: (B, H, W, C)
        B = tf.shape(x)[0]
        gap = self.gap(x)                                   
        gap = tf.reshape(gap, [B, self.channels, 1])        
        attn = tf.sigmoid(self.conv(gap))                   
        attn = tf.reshape(attn, [B, 1, 1, self.channels]) 
        return x * attn                                   
    def get_config(self):
        cfg = super().get_config()
        cfg['kernel_size'] = self.kernel_size
        return cfg

# ← NEW: Pixel Attention (spatial attention — learns WHERE to focus)
class PixelAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    def build(self, input_shape):
        C = input_shape[-1]
        self.conv1 = layers.Conv2D(max(8, C // 4), 1, padding='same', use_bias=False)
        self.conv2 = layers.Conv2D(1, 1, padding='same', activation='sigmoid')
    def call(self, x):
        attn = tf.nn.relu(self.conv1(x))
        attn = self.conv2(attn)                              # (B, H, W, 1)
        return x * attn
    def get_config(self):
        return super().get_config()

class PhysicsCorrectionLayer(layers.Layer):
    def __init__(self, eps=0.1, blend=0.08, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps
        self.blend = blend
    def call(self, inputs):
        img, A, t = inputs
        # img: (B,H,W,3), A: (B,3), t: (B,H,W,1)
        A_bc = tf.reshape(A, [-1, 1, 1, 3])                 # (B,1,1,3)
        A_bc = tf.broadcast_to(A_bc, tf.shape(img))          # (B,H,W,3)
        t_bc = tf.broadcast_to(t, tf.shape(img))             # (B,H,W,3)
        j = (img - A_bc) / (t_bc + self.eps) + A_bc          # (B,H,W,3)
        j = tf.clip_by_value(j, 0.0, 1.0)
        return img * (1 - self.blend) + j * self.blend        # (B,H,W,3)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'eps': self.eps, 'blend': self.blend})
        return cfg

def dws_conv_block(x, filters, kernel_size=3, strides=1):
    # (B,H,W,Cin) → DepthwiseConv → (B,H,W,Cin) → 1x1 Conv → (B,H,W,filters)
    x = layers.DepthwiseConv2D(kernel_size, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu6')(x)
    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu6')(x)
    return x

def residual_block(x, filters, use_eca=True, drop_rate=SPATIAL_DROP):
    skip = x
    if x.shape[-1] != filters:
        skip = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
    x = dws_conv_block(x, filters)
    x = dws_conv_block(x, filters)
    if use_eca:
        x = ECABlock(kernel_size=ECA_KERNEL)(x)
        x = PixelAttention()(x)                              # ← NEW: spatial attention after ECA
    if drop_rate > 0:
        x = layers.SpatialDropout2D(drop_rate)(x)
    x = layers.Add()([x, skip])
    return x

def encoder_block(x, filters):
    x = residual_block(x, filters, use_eca=True, drop_rate=SPATIAL_DROP)
    skip = x
    x = layers.MaxPooling2D(2)(x)
    return x, skip

def decoder_block(x, skip, filters):
    x = layers.UpSampling2D(2, interpolation='bilinear')(x)
    x = layers.Concatenate()([x, skip])
    x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu6')(x)
    x = residual_block(x, filters, use_eca=True, drop_rate=SPATIAL_DROP/2)
    return x

print("Building blocks defined (Dual Attention: ECA + PixelAttention)")


## Full ECA-LDNet Architecture

In [ ]:

# CELL 7 — FULL ECA-LDNet ARCHITECTURE

def build_eca_ldnet(img_size=IMG_SIZE):
    inputs = layers.Input(shape=(img_size, img_size, CHANNELS), name='hazy_input')

    # ENCODER
    x, skip1 = encoder_block(inputs, ENC_FILTERS[0])  # skip1=256x256x32, x=128x128x32
    x, skip2 = encoder_block(x, ENC_FILTERS[1])       # skip2=128x128x64, x=64x64x64
    x, skip3 = encoder_block(x, ENC_FILTERS[2])       # skip3=64x64x128,  x=32x32x128
    x, skip4 = encoder_block(x, ENC_FILTERS[3])       # skip4=32x32x256,  x=16x16x256

    # BOTTLENECK (16x16x512, NO MaxPool)
    x = dws_conv_block(x, ENC_FILTERS[4])              # 16x16x512
    x = residual_block(x, ENC_FILTERS[4], use_eca=True, drop_rate=0)
    x = layers.Dropout(BOTTLENECK_DROP)(x)
    bottleneck = x                                      # 16x16x512

    # PHYSICS GUIDANCE MODULE
    A = layers.GlobalAveragePooling2D()(bottleneck)     # (512,)
    A = layers.Dense(64, activation='relu')(A)
    A = layers.Dropout(0.1)(A)
    A = layers.Dense(32, activation='relu')(A)
    A = layers.Dense(3, activation='sigmoid', dtype='float32', name='atm_light')(A)  # (3,)

    t = layers.Conv2D(32, 1, padding='same')(bottleneck)
    t = layers.BatchNormalization()(t)
    t = layers.Activation('relu')(t)
    t = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', padding='same', name='transmission')(t)  # 16x16x1

    t_16 = layers.UpSampling2D(2, interpolation='bilinear')(t)   # 32x32x1
    t_32 = layers.UpSampling2D(4, interpolation='bilinear')(t)   # 64x64x1

    # DECODER with physics attention
    x = decoder_block(x, skip4, DEC_FILTERS[0])        # Up 16→32, cat skip4(32x32x256) → 32x32x256
    hm16 = 1.0 - t_16                                  # 32x32x1
    x = layers.Multiply(name='phys_d4')([x, hm16])     # 32x32x256 * 32x32x1 → broadcasts

    x = decoder_block(x, skip3, DEC_FILTERS[1])        # Up 32→64, cat skip3(64x64x128) → 64x64x128
    hm32 = 1.0 - t_32                                  # 64x64x1
    x = layers.Multiply(name='phys_d3')([x, hm32])     # 64x64x128 * 64x64x1 → broadcasts

    x = decoder_block(x, skip2, DEC_FILTERS[2])        # Up 64→128, cat skip2(128x128x64) → 128x128x64
    x = decoder_block(x, skip1, DEC_FILTERS[3])        # Up 128→256, cat skip1(256x256x32) → 256x256x32

    # Final refinement (NO extra UpSampling)
    x = dws_conv_block(x, 16)                           # 256x256x16

    # OUTPUT
    raw = layers.Conv2D(3, 1, padding='same', use_bias=False, dtype='float32', name='raw_conv')(x)  # 256x256x3
    raw = layers.Activation('sigmoid', dtype='float32', name='raw_sigmoid')(raw)                     # 256x256x3

    t_full = layers.UpSampling2D(img_size // 16, interpolation='bilinear', name='t_full')(t)  # 16*16=256x256x1
    output = PhysicsCorrectionLayer(eps=0.1, blend=0.08, dtype='float32', name='physics_output')([raw, A, t_full])  # 256x256x3

    return Model(inputs=inputs, outputs=output, name='ECA_LDNet')

tf.keras.backend.clear_session()
model = build_eca_ldnet()
total_params = model.count_params()
print(f"\n{'='*60}")
print(f"ECA-LDNet — {total_params:,} params ({total_params/1e6:.3f}M)")
print(f"{'='*60}")
model.summary(line_length=100)

with open(f'{METRIC_DIR}/model_summary.txt', 'w') as f:
    model.summary(print_fn=lambda s: f.write(s+'\n'))
    f.write(f'\nTotal: {total_params:,} ({total_params/1e6:.3f}M)\n')


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8 — LOSS FUNCTIONS (with VGG Perceptual Loss)
# ══════════════════════════════════════════════════════════════
from tensorflow.keras.applications import VGG16

# --- VGG16 Perceptual Feature Extractor (frozen, adds 0 trainable params) ---
_vgg_base = VGG16(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
_vgg_base.trainable = False
perceptual_model = Model(
    inputs=_vgg_base.input,
    outputs=[_vgg_base.get_layer('block1_conv2').output,
             _vgg_base.get_layer('block2_conv2').output,
             _vgg_base.get_layer('block3_conv3').output],
    name='vgg_perceptual'
)
perceptual_model.trainable = False

# --- Helper ---
def cast_f32(a, b):
    return tf.cast(a, tf.float32), tf.cast(b, tf.float32)

# --- Individual Losses ---
def mae_loss(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    return tf.reduce_mean(tf.abs(y_true - y_pred))

def charbonnier_loss(y_true, y_pred, eps=1e-3):
    y_true, y_pred = cast_f32(y_true, y_pred)
    return tf.reduce_mean(tf.sqrt(tf.square(y_true - y_pred) + eps*eps))

def ssim_loss(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    return 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def edge_loss(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    return tf.reduce_mean(tf.abs(tf.image.sobel_edges(y_true) - tf.image.sobel_edges(y_pred)))

def frequency_loss(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    def log_fft(img):
        gray = tf.reduce_mean(img, axis=-1)
        return tf.math.log1p(tf.abs(tf.signal.rfft2d(gray)))
    return tf.reduce_mean(tf.abs(log_fft(y_true) - log_fft(y_pred)))

def perceptual_loss(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    mean = tf.constant([123.68, 116.779, 103.939], shape=[1, 1, 1, 3], dtype=tf.float32)
    true_feats = perceptual_model(y_true * 255.0 - mean, training=False)
    pred_feats = perceptual_model(y_pred * 255.0 - mean, training=False)
    loss = 0.0
    for t_f, p_f in zip(true_feats, pred_feats):
        loss += tf.reduce_mean(tf.abs(t_f - p_f))
    return loss / 3.0

# --- Combined Losses (S1/S2 and S3) ---
def combined_loss(y_true, y_pred):
    return (W_MAE    * charbonnier_loss(y_true, y_pred)
          + W_SSIM   * ssim_loss(y_true, y_pred)
          + W_EDGE   * edge_loss(y_true, y_pred)
          + W_FREQ   * frequency_loss(y_true, y_pred)
          + W_PERCEP * perceptual_loss(y_true, y_pred))

def combined_loss_s3(y_true, y_pred):
    return (0.40 * charbonnier_loss(y_true, y_pred)
          + 0.20 * ssim_loss(y_true, y_pred)
          + 0.10 * edge_loss(y_true, y_pred)
          + 0.05 * frequency_loss(y_true, y_pred)
          + 0.25 * perceptual_loss(y_true, y_pred))

# --- Metrics ---
def psnr_metric(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def ssim_metric(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    return tf.image.ssim(y_true, y_pred, max_val=1.0)

def mae_metric(y_true, y_pred):
    y_true, y_pred = cast_f32(y_true, y_pred)
    return tf.reduce_mean(tf.abs(y_true - y_pred))

print("Loss functions defined (VGG Perceptual active)")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9 — LR SCHEDULE & CALLBACKS
# ══════════════════════════════════════════════════════════════
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps, min_lr=MIN_LR):
        self.peak_lr = float(peak_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps = float(total_steps)
        self.min_lr = float(min_lr)
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / (self.warmup_steps + 1e-8))
        progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps + 1e-8)
        cos_lr = self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (
            1.0 + tf.cos(3.14159265 * tf.clip_by_value(progress, 0.0, 1.0)))
        return tf.where(step < self.warmup_steps, warmup_lr, cos_lr)
    def get_config(self):
        return {'peak_lr': self.peak_lr, 'warmup_steps': self.warmup_steps,
                'total_steps': self.total_steps, 'min_lr': self.min_lr}

class CSVMetricsLogger(callbacks.Callback):
    def __init__(self, path, stage):
        super().__init__()
        self.path = path; self.stage = stage; self.records = []
    def on_epoch_end(self, epoch, logs=None):
        row = {'epoch': epoch, 'stage': self.stage}
        row.update(logs or {})
        try:
            opt = self.model.optimizer
            lr = opt.learning_rate
            row['lr'] = float(lr(opt.iterations) if callable(lr) else lr)
        except: row['lr'] = 0.0
        self.records.append(row)
        pd.DataFrame(self.records).to_csv(self.path, index=False)

class SampleVizCallback(callbacks.Callback):
    def __init__(self, val_ds, save_dir, every_n=10):
        super().__init__()
        self.val_ds = val_ds; self.save_dir = save_dir; self.every_n = every_n
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n != 0: return
        for hb, gb in self.val_ds.take(1):
            pb = self.model.predict(hb[:4], verbose=0)
            fig, axes = plt.subplots(4, 3, figsize=(12, 16))
            for i in range(min(4, len(hb))):
                axes[i,0].imshow(np.clip(hb[i].numpy(),0,1)); axes[i,0].set_title('Hazy'); axes[i,0].axis('off')
                axes[i,1].imshow(np.clip(pb[i],0,1)); axes[i,1].set_title('Pred'); axes[i,1].axis('off')
                axes[i,2].imshow(np.clip(gb[i].numpy(),0,1)); axes[i,2].set_title('GT'); axes[i,2].axis('off')
            plt.suptitle(f'Epoch {epoch+1}')
            plt.tight_layout()
            plt.savefig(f'{self.save_dir}/ep{epoch+1:03d}.png', dpi=80, bbox_inches='tight')
            plt.close()

def make_callbacks(stage, lr, warmup_ep, total_ep, steps):
    total_steps = total_ep * steps
    warmup_steps = warmup_ep * steps
    lr_schedule = WarmupCosineDecay(lr, warmup_steps, total_steps)
    cb = [
        callbacks.ModelCheckpoint(
            f'{MODEL_DIR}/stage{stage}_best.keras',
            monitor='val_psnr_metric', mode='max', save_best_only=True, verbose=1),
        callbacks.EarlyStopping(
            monitor='val_psnr_metric', mode='max',
            patience=PATIENCE_ES, restore_best_weights=True, verbose=1),

        CSVMetricsLogger(f'{HISTORY_DIR}/stage{stage}_history.csv', stage),
        SampleVizCallback(val_ds, PLOT_DIR, every_n=10),
    ]
    return lr_schedule, cb

print("LR schedule and callbacks defined")

## Stage 1 — Full Model Training (60 epochs, LR=1e-3)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10 — STAGE 1 TRAINING
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STAGE 1 — Full Model Training")
# print(f"Epochs: {EPOCHS_S+21} | LR: {LR_S1} | Warmup: {WARMUP_S1} ep")
# print("="*60)

lr_s1, cbs_s1 = make_callbacks(1, LR_S1, WARMUP_S1, EPOCHS_S1, STEPS_PER_EPOCH)

optimizer_s1 = tf.keras.optimizers.AdamW(
    learning_rate=lr_s1, weight_decay=1e-4, clipnorm=1.0)

model.load_weights(f'/kaggle/input/datasets/codeninjalucky/stage1-best-weights/stage1_best.keras')

model.compile(optimizer=optimizer_s1, loss=combined_loss,
              metrics=[psnr_metric, ssim_metric, mae_metric])

t0 = time.time()
history1 = model.fit(train_ds, epochs=EPOCHS_S1, validation_data=val_ds,
                     callbacks=cbs_s1, verbose=1,initial_epoch=21)
t1 = time.time()

best_s1 = max(history1.history.get('val_psnr_metric', [0]))
print(f"\nStage 1: {(t1-t0)/3600:.2f}h | Best val PSNR: {best_s1:.4f} dB")
gc.collect()

## Stage 2 — Fine-tuning (30 epochs, LR=1e-4)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — STAGE 2 TRAINING
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STAGE 2 — Fine-tuning")
print(f"Epochs: {EPOCHS_S2} | LR: {LR_S2} | Warmup: {WARMUP_S2} ep")
print("="*60)

model.load_weights(f'{MODEL_DIR}/stage1_best.keras')
print("Stage 1 best weights loaded")

lr_s2, cbs_s2 = make_callbacks(2, LR_S2, WARMUP_S2, EPOCHS_S2, STEPS_PER_EPOCH)

optimizer_s2 = tf.keras.optimizers.AdamW(
    learning_rate=lr_s2, weight_decay=5e-5, clipnorm=1.0)

model.compile(optimizer=optimizer_s2, loss=combined_loss,
              metrics=[psnr_metric, ssim_metric, mae_metric])

t0 = time.time()
history2 = model.fit(train_ds, epochs=EPOCHS_S2, validation_data=val_ds,
                     callbacks=cbs_s2, verbose=1)
t1 = time.time()

best_s2 = max(history2.history.get('val_psnr_metric', [0]))
print(f"\nStage 2: {(t1-t0)/3600:.2f}h | Best val PSNR: {best_s2:.4f} dB")
gc.collect()

## Stage 3 — Final Polish (20 epochs, LR=5e-5, batch=8)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 12 — STAGE 3 TRAINING
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STAGE 3 — Final Polish")
print(f"Epochs: {EPOCHS_S3} | LR: {LR_S3} | Batch: {BATCH_SIZE_S3}")
print("="*60)

model.load_weights(f'{MODEL_DIR}/stage2_best.keras')
print("Stage 2 best weights loaded")

STEPS_S3 = len(tr_h) // BATCH_SIZE_S3
train_ds3 = build_dataset(tr_h, tr_g, augment=True, batch=BATCH_SIZE_S3)
val_ds3 = build_dataset(va_h, va_g, augment=False, batch=BATCH_SIZE_S3, shuffle=False, cache_ds=True)

lr_s3, cbs_s3 = make_callbacks(3, LR_S3, 0, EPOCHS_S3, STEPS_S3)

optimizer_s3 = tf.keras.optimizers.AdamW(
    learning_rate=lr_s3, weight_decay=2.5e-5, clipnorm=0.5)

model.compile(optimizer=optimizer_s3, loss=combined_loss_s3,
              metrics=[psnr_metric, ssim_metric, mae_metric])

t0 = time.time()
history3 = model.fit(train_ds3, epochs=EPOCHS_S3, validation_data=val_ds3,
                     callbacks=cbs_s3, verbose=1)
t1 = time.time()

best_s3 = max(history3.history.get('val_psnr_metric', [0]))
print(f"\nStage 3: {(t1-t0)/3600:.2f}h | Best val PSNR: {best_s3:.4f} dB")

# Load absolute best
best_stage = int(np.argmax([best_s1, best_s2, best_s3])) + 1
model.load_weights(f'{MODEL_DIR}/stage{best_stage}_best.keras')
print(f"Loaded best from Stage {best_stage} (PSNR={max(best_s1,best_s2,best_s3):.4f})")
gc.collect()

## Save All Artifacts

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 13 — SAVE ALL ARTIFACTS
# ══════════════════════════════════════════════════════════════
model.save(f'{MODEL_DIR}/stage3_final.keras')
model.save_weights(f'{MODEL_DIR}/weights_final.h5')
print("Models saved")

def h2df(hist, stage):
    df = pd.DataFrame(hist.history)
    df['stage'] = stage
    df['epoch'] = range(len(df))
    return df

df1, df2, df3 = h2df(history1,1), h2df(history2,2), h2df(history3,3)
df_all = pd.concat([df1, df2, df3], ignore_index=True)
df_all.to_csv(f'{HISTORY_DIR}/full_history.csv', index=False)
print("History saved")

# Training curves
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
colors = {'1':('#2196F3','#1565C0'), '2':('#4CAF50','#1B5E20'), '3':('#FF9800','#E65100')}
for df, s in [(df1,'1'),(df2,'2'),(df3,'3')]:
    c1, c2 = colors[s]
    ep = df['epoch'].values
    if 'loss' in df.columns:
        axes[0,0].plot(ep, df['loss'], color=c1, label=f'Train S{s}')
        axes[0,0].plot(ep, df['val_loss'], color=c2, ls='--', label=f'Val S{s}')
    if 'psnr_metric' in df.columns:
        axes[0,1].plot(ep, df['psnr_metric'], color=c1, label=f'Train S{s}')
        axes[0,1].plot(ep, df['val_psnr_metric'], color=c2, ls='--', label=f'Val S{s}')
    if 'ssim_metric' in df.columns:
        axes[1,0].plot(ep, df['ssim_metric'], color=c1, label=f'Train S{s}')
        axes[1,0].plot(ep, df['val_ssim_metric'], color=c2, ls='--', label=f'Val S{s}')
    if 'mae_metric' in df.columns:
        axes[1,1].plot(ep, df['mae_metric'], color=c1, label=f'Train S{s}')
        axes[1,1].plot(ep, df['val_mae_metric'], color=c2, ls='--', label=f'Val S{s}')
for ax, t in zip(axes.flat, ['Loss','PSNR (dB)','SSIM','MAE']):
    ax.set_title(t, fontsize=13, fontweight='bold')
    ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.suptitle('ECA-LDNet Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plots saved")

## Inference Utilities

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 14 — INFERENCE UTILITIES
# ══════════════════════════════════════════════════════════════
def gaussian_window_2d(size, sigma=0.35):
    c = np.linspace(-1, 1, size)
    g = np.exp(-(c**2) / (2*sigma**2))
    w = np.outer(g, g)
    return (w / w.max()).astype(np.float32)

def sliding_window_predict(model, img, patch=IMG_SIZE, stride=64, batch_size=8):
    H, W = img.shape[:2]
    output = np.zeros((H,W,3), np.float32)
    weight = np.zeros((H,W,3), np.float32)
    win3d = gaussian_window_2d(patch)[:,:,np.newaxis]
    ys = list(range(0, max(H-patch,1), stride))
    xs = list(range(0, max(W-patch,1), stride))
    if not ys or ys[-1] != max(H-patch,0): ys.append(max(H-patch,0))
    if not xs or xs[-1] != max(W-patch,0): xs.append(max(W-patch,0))
    coords = [(y,x) for y in ys for x in xs]
    crops = np.stack([img[y:y+patch, x:x+patch] for y,x in coords]).astype(np.float32)
    preds = model.predict(crops, batch_size=batch_size, verbose=0)
    for pred_p, (y,x) in zip(preds, coords):
        output[y:y+patch, x:x+patch] += pred_p * win3d
        weight[y:y+patch, x:x+patch] += win3d
    return np.clip(output / (weight + 1e-8), 0, 1)

def predict_full(model, img, patch=IMG_SIZE, stride=64, batch_size=8, tta=True):
    H, W = img.shape[:2]
    def _pred(im):
        if H <= patch and W <= patch:
            return model.predict(im[np.newaxis], verbose=0)[0].astype(np.float32)
        return sliding_window_predict(model, im, patch, stride, batch_size)
    out = _pred(img)
    if tta:
        o_lr = _pred(img[:,::-1,:])[:,::-1,:]
        o_ud = _pred(img[::-1,:,:])[::-1,:,:]
        o_r  = np.rot90(_pred(np.rot90(img,1)), -1)
        out  = (out + o_lr + o_ud + o_r) / 4.0
    return np.clip(out, 0, 1).astype(np.float32)

print("Inference utilities defined")

## Full Evaluation on All Test Sets

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 15 — EVALUATION
# ══════════════════════════════════════════════════════════════
def evaluate_dataset(model, hazy_dir, gt_dir, name, limit=None, tta=True, save_csv=None):
    if not hazy_dir or not gt_dir:
        print(f"  SKIP {name} — not found")
        return None
    files = sorted([f for f in os.listdir(hazy_dir) if f.lower().endswith(('.png','.jpg','.jpeg','.bmp'))])
    if limit: files = files[:limit]
    print(f"\nEvaluating [{name}] — {len(files)} images, TTA={tta}")
    records = []
    for i, fname in enumerate(files):
        hp = os.path.join(hazy_dir, fname)
        gp = get_gt_path(fname, gt_dir)
        if not gp: continue
        hazy, gt = load_image_pair(hp, gp)
        if hazy is None: continue
        pred = predict_full(model, hazy, tta=tta)
        p = float(tf.image.psnr(pred[np.newaxis], gt[np.newaxis], max_val=1.0))
        s = float(tf.image.ssim(pred[np.newaxis], gt[np.newaxis], max_val=1.0))
        m = float(np.mean(np.abs(pred - gt)))
        records.append({'file': fname, 'PSNR': p, 'SSIM': s, 'MAE': m})
        if (i+1) % 100 == 0:
            print(f"  [{i+1}/{len(files)}] PSNR: {np.mean([r['PSNR'] for r in records]):.4f}")
    df = pd.DataFrame(records)
    if len(df) > 0:
        print(f"  {name}: PSNR={df.PSNR.mean():.4f}+-{df.PSNR.std():.4f}  SSIM={df.SSIM.mean():.4f}")
    if save_csv: df.to_csv(save_csv, index=False)
    return df

all_results = {}
df_r6k = evaluate_dataset(model, R6K_TEST_HAZY, R6K_TEST_GT, 'RESIDE-6K', 1000, True,
                           f'{METRIC_DIR}/results_reside6k.csv')
if df_r6k is not None: all_results['RESIDE-6K'] = df_r6k

df_si = evaluate_dataset(model, SOTS_IN_HAZY, SOTS_IN_GT, 'SOTS Indoor', 500, True,
                          f'{METRIC_DIR}/results_sots_indoor.csv')
if df_si is not None: all_results['SOTS Indoor'] = df_si

df_so = evaluate_dataset(model, SOTS_OUT_HAZY, SOTS_OUT_GT, 'SOTS Outdoor', 500, True,
                          f'{METRIC_DIR}/results_sots_outdoor.csv')
if df_so is not None: all_results['SOTS Outdoor'] = df_so

## Visual Comparison Plots

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 16 — VISUAL COMPARISONS
# ══════════════════════════════════════════════════════════════
def plot_comparison(model, hazy_dir, gt_dir, save_path, title='Results', tta=False):
    if not hazy_dir or not gt_dir: return
    files = sorted(os.listdir(hazy_dir))
    random.shuffle(files)
    samples = []
    for fname in files[:100]:
        if not fname.lower().endswith(('.png','.jpg','.jpeg','.bmp')): continue
        gp = get_gt_path(fname, gt_dir)
        if not gp: continue
        hazy, gt = load_image_pair(os.path.join(hazy_dir, fname), gp)
        if hazy is None: continue
        pred = predict_full(model, hazy, tta=tta)
        pv = float(tf.image.psnr(pred[np.newaxis], gt[np.newaxis], max_val=1.0))
        sv = float(tf.image.ssim(pred[np.newaxis], gt[np.newaxis], max_val=1.0))
        samples.append((pv, sv, hazy, pred, gt))
        if len(samples) >= 30: break
    if len(samples) < 6:
        print(f"Not enough samples for {title}")
        return
    samples.sort(key=lambda x: x[0])
    show = samples[:3] + samples[-3:]
    labels = ['Worst']*3 + ['Best']*3
    fig, axes = plt.subplots(6, 4, figsize=(20, 28))
    for row, (pv, sv, hazy, pred, gt) in enumerate(show):
        error = np.abs(pred - gt)
        axes[row,0].imshow(np.clip(hazy,0,1)); axes[row,0].set_title(f'[{labels[row]}] Hazy'); axes[row,0].axis('off')
        axes[row,1].imshow(np.clip(pred,0,1)); axes[row,1].set_title(f'Pred PSNR={pv:.2f}'); axes[row,1].axis('off')
        axes[row,2].imshow(np.clip(gt,0,1)); axes[row,2].set_title('GT'); axes[row,2].axis('off')
        im = axes[row,3].imshow(error, cmap='hot', vmin=0, vmax=0.3); axes[row,3].set_title('Error'); axes[row,3].axis('off')
        plt.colorbar(im, ax=axes[row,3], fraction=0.046, pad=0.04)
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout(); plt.savefig(save_path, dpi=120, bbox_inches='tight'); plt.show()

plot_comparison(model, R6K_TEST_HAZY, R6K_TEST_GT, f'{RESULT_DIR}/visual_reside6k.png',
                'ECA-LDNet — RESIDE-6K Results')
plot_comparison(model, SOTS_IN_HAZY, SOTS_IN_GT, f'{RESULT_DIR}/visual_sots_indoor.png',
                'ECA-LDNet — SOTS Indoor Results')
plot_comparison(model, SOTS_OUT_HAZY, SOTS_OUT_GT, f'{RESULT_DIR}/visual_sots_outdoor.png',
                'ECA-LDNet — SOTS Outdoor Results')

## Pixel Distribution & Quality Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 17 — PIXEL DISTRIBUTION & SHARPNESS
# ══════════════════════════════════════════════════════════════
def plot_pixel_distribution(model, hazy_dir, gt_dir, save_path, n_samples=50):
    if not hazy_dir or not gt_dir: return
    files = sorted(os.listdir(hazy_dir))[:n_samples]
    hazy_all, pred_all, gt_all = [], [], []
    for fname in files:
        if not fname.lower().endswith(('.png','.jpg','.jpeg','.bmp')): continue
        gp = get_gt_path(fname, gt_dir)
        if not gp: continue
        hazy, gt = load_image_pair(os.path.join(hazy_dir, fname), gp)
        if hazy is None: continue
        pred = predict_full(model, hazy, tta=False)
        hazy_all.append(hazy); pred_all.append(pred); gt_all.append(gt)
    if len(hazy_all) == 0: return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    ax1.hist(np.concatenate([h.flatten() for h in hazy_all]), bins=60, alpha=0.5, label='Hazy', color='orange')
    ax1.hist(np.concatenate([p.flatten() for p in pred_all]), bins=60, alpha=0.5, label='Predicted', color='blue')
    ax1.hist(np.concatenate([g.flatten() for g in gt_all]), bins=60, alpha=0.5, label='GT', color='green')
    ax1.set_title('Pixel Distribution'); ax1.legend(); ax1.grid(alpha=0.3)
    def sharp(imgs):
        scores = []
        for im in imgs[:10]:
            gray = cv2.cvtColor((im*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
            scores.append(cv2.Laplacian(gray, cv2.CV_64F).var())
        return np.mean(scores)
    vals = [sharp(hazy_all), sharp(pred_all), sharp(gt_all)]
    bars = ax2.bar(['Hazy','Predicted','GT'], vals, color=['orange','blue','green'], alpha=0.8)
    ax2.bar_label(bars, fmt='%.1f'); ax2.set_title('Sharpness'); ax2.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=120, bbox_inches='tight'); plt.show()

plot_pixel_distribution(model, R6K_TEST_HAZY, R6K_TEST_GT, f'{RESULT_DIR}/pixel_analysis.png')

## Final Report & Paper Tables

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 18 — FINAL METRICS REPORT
# ══════════════════════════════════════════════════════════════
# Inference speed
dummy = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
_ = model.predict(dummy, verbose=0)
timings = []
for _ in range(30):
    t0 = time.time(); model.predict(dummy, verbose=0); timings.append(time.time()-t0)
avg_ms = np.mean(timings)*1000
fps = 1000/avg_ms

# Get best results
def get_mean(name, col):
    if name in all_results and len(all_results[name]) > 0:
        return all_results[name][col].mean()
    return 0.0

my_psnr = get_mean('SOTS Indoor','PSNR') or get_mean('RESIDE-6K','PSNR')
my_ssim = get_mean('SOTS Indoor','SSIM') or get_mean('RESIDE-6K','SSIM')

comparison = pd.DataFrame({
    'Method': ['DCP','DehazeNet','MSCNN','AOD-Net','GFN','GridDehazeNet','FFA-Net','ECA-LDNet (Ours)'],
    'Year': [2009,2016,2016,2017,2018,2019,2020,2025],
    'PSNR': [16.62,21.14,17.57,19.06,21.55,32.16,36.39,round(my_psnr,2)],
    'SSIM': [0.818,0.847,0.810,0.850,0.844,0.984,0.989,round(my_ssim,4)],
    'Params': ['—','0.009M','0.008M','0.002M','0.530M','0.956M','4.456M',f'{model.count_params()/1e6:.3f}M'],
})
comparison.to_csv(f'{METRIC_DIR}/comparison_table.csv', index=False)

with open(f'{METRIC_DIR}/final_report.txt', 'w') as f:
    f.write("ECA-LDNet — Final Report\n" + "="*60 + "\n")
    f.write(f"Parameters: {model.count_params():,} ({model.count_params()/1e6:.3f}M)\n")
    f.write(f"Inference: {avg_ms:.1f} ms ({fps:.1f} FPS)\n\n")
    for name, df in all_results.items():
        f.write(f"{name} ({len(df)} imgs): PSNR={df.PSNR.mean():.4f} SSIM={df.SSIM.mean():.4f}\n")
    f.write("\n" + comparison.to_string(index=False))

print("\n" + "="*65)
print("FINAL SUMMARY — ECA-LDNet")
print("="*65)
print(f"  Params: {model.count_params():,} ({model.count_params()/1e6:.3f}M)")
print(f"  Speed:  {avg_ms:.1f} ms ({fps:.1f} FPS)")
for name, df in all_results.items():
    print(f"  {name:<15}: PSNR={df.PSNR.mean():.4f} SSIM={df.SSIM.mean():.4f}")
print("\nComparison:")
print(comparison.to_string(index=False))
print("="*65)
print("TRAINING COMPLETE")

## Offline Testing Script (Copy to local machine)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 19 — OFFLINE TEST SCRIPT (save as test_offline.py locally)
# ══════════════════════════════════════════════════════════════
offline_script = '''
#!/usr/bin/env python3
"""ECA-LDNet Offline Testing — run on local machine after downloading model"""
import os, cv2, sys, numpy as np, argparse
import tensorflow as tf

IMG_SIZE = 256

# Register custom objects
class ECABlock(tf.keras.layers.Layer):
    def __init__(self, kernel_size=3, **kw):
        super().__init__(**kw); self.kernel_size = kernel_size
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.conv = tf.keras.layers.Conv1D(1, self.kernel_size, padding="same", use_bias=False)
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
    def call(self, x):
        B = tf.shape(x)[0]; gap = self.gap(x)
        gap = tf.reshape(gap, [B, 1, self.channels])
        attn = tf.sigmoid(self.conv(gap))
        return x * tf.reshape(attn, [B, 1, 1, self.channels])
    def get_config(self):
        c = super().get_config(); c["kernel_size"] = self.kernel_size; return c

class PhysicsCorrectionLayer(tf.keras.layers.Layer):
    def __init__(self, eps=0.1, blend=0.08, **kw):
        super().__init__(**kw); self.eps = eps; self.blend = blend
    def call(self, inputs):
        img, A, t = inputs
        A_bc = tf.broadcast_to(tf.reshape(A, [-1,1,1,3]), tf.shape(img))
        t_bc = tf.broadcast_to(t, tf.shape(img))
        j = tf.clip_by_value((img - A_bc) / (t_bc + self.eps) + A_bc, 0, 1)
        return img * (1 - self.blend) + j * self.blend
    def get_config(self):
        c = super().get_config(); c.update({"eps": self.eps, "blend": self.blend}); return c

def preprocess(img):
    img = np.power(np.clip(img, 0, 1), 0.9).astype(np.float32)
    for c in range(3):
        lo, hi = img[:,:,c].min(), img[:,:,c].max()
        if hi > lo + 1e-6: img[:,:,c] = (img[:,:,c] - lo) / (hi - lo)
    return np.clip(img, 0, 1).astype(np.float32)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", default="dehazed_output.png")
    args = parser.parse_args()

    custom = {"ECABlock": ECABlock, "PhysicsCorrectionLayer": PhysicsCorrectionLayer}
    model = tf.keras.models.load_model(args.model, custom_objects=custom, compile=False)
    print(f"Model loaded: {model.count_params():,} params")

    img = cv2.imread(args.input)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
    img = preprocess(img)

    pred = model.predict(img[np.newaxis], verbose=0)[0]
    pred = np.clip(pred, 0, 1)

    out = cv2.cvtColor((pred * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
    cv2.imwrite(args.output, out)
    print(f"Saved: {args.output}")

if __name__ == "__main__":
    main()
'''

with open(f'{SAVE_DIR}/test_offline.py', 'w') as f:
    f.write(offline_script)
print("Offline test script saved to /kaggle/working/test_offline.py")
print("Download and run: python test_offline.py --model models/stage3_final.keras --input hazy.jpg")